# Stellan runtime: Lite and Max CPU checks

Run this in a **separate Kaggle notebook after your current benchmark has finished**. Turn Internet on, then choose **Run All**. No GPU is needed: the new engine has no GPU backend yet. The notebook builds both engines, verifies public model hashes and writes paired CPU results. Max is skipped below 24 GiB reported RAM; use a larger machine rather than bypassing that check.

This is development evidence, not a claim that the new engine is better. It does not change the app, push to main, publish a release or contact other projects. Build tools and weights are downloaded, but prompts run locally. Only small reports and logs remain after the temporary build/model folder is cleaned up.


In [ ]:
import pathlib, tempfile, subprocess, sys, os, hashlib, urllib.request, tarfile, json
results = pathlib.Path('/kaggle/working/stellan-runtime-results')
results.mkdir(parents=True, exist_ok=True)
workspace = tempfile.TemporaryDirectory(prefix='stellan-cpu-', dir='/kaggle/working')
root = pathlib.Path(workspace.name)
def run(args, logfile):
    with open(results / logfile, 'ab') as output:
        subprocess.run([str(value) for value in args], check=True, stdout=output, stderr=subprocess.STDOUT, env=environment)
def download(url, target, expected):
    digest = hashlib.sha256()
    size = 0
    with urllib.request.urlopen(url, timeout=120) as source, open(target, 'wb') as destination:
        while True:
            chunk = source.read(8 * 1024 * 1024)
            if not chunk:
                break
            destination.write(chunk)
            digest.update(chunk)
            size += len(chunk)
            if size % (512 * 1024 * 1024) == 0:
                print(f'Downloaded {size // (1024 * 1024)} MB', flush=True)
    if digest.hexdigest() != expected:
        raise RuntimeError('The downloaded file did not pass its integrity check; run this cell again')
environment = dict(os.environ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--target', str(root / 'tools'), 'cmake'], check=True)
cmake = root / 'tools/cmake/data/bin/cmake'
archive = root / 'node.tar.gz'
download('https://nodejs.org/dist/v24.22.1/node-v24.22.1-linux-x64.tar.gz', archive, 'eb6728e77c576a35162c122c45ccadf224d89ff00abc8dae46cc2897c6002d7c')
with tarfile.open(archive) as source:
    source.extractall(root, filter='data')
node = root / 'node-v24.22.1-linux-x64/bin/node'
environment['PATH'] = str(node.parent) + os.pathsep + environment.get('PATH', '')
project = root / 'stick_ai'
run(['git', 'clone', '--depth', '1', '--branch', 'experimental/stellan-runtime', 'https://github.com/alfredmurray-goat/stick_ai.git', project], 'build.log')
source = project / 'experimental/stellan-runtime'
run([cmake, '-S', source, '-B', root / 'native-build', '-DCMAKE_BUILD_TYPE=Release'], 'build.log')
run([cmake, '--build', root / 'native-build', '--parallel', '2'], 'build.log')
run([root / 'tools/cmake/data/bin/ctest', '--test-dir', root / 'native-build', '--output-on-failure'], 'tests.log')
run([node, '--test', source / 'tests/node-checks.mjs'], 'tests.log')
print('The independent runtime built and its tests passed')


In [ ]:
reference = root / 'reference'
reference_build = root / 'reference-build'
run(['git', 'init', reference], 'build.log')
run(['git', '-C', reference, 'remote', 'add', 'origin', 'https://github.com/ggml-org/llama.cpp.git'], 'build.log')
run(['git', '-C', reference, 'fetch', '--depth', '1', 'origin', 'cd959bae99ff4bed22bef2604f45241c09bbb8c8'], 'build.log')
run(['git', '-C', reference, 'checkout', '--detach', 'FETCH_HEAD'], 'build.log')
run(['git', '-C', reference, 'apply', project / 'scripts/llama-split-dir.patch'], 'build.log')
run([cmake, '-S', reference, '-B', reference_build, '-DCMAKE_BUILD_TYPE=Release', '-DBUILD_SHARED_LIBS=OFF', '-DGGML_NATIVE=OFF', '-DGGML_OPENMP=OFF', '-DGGML_BLAS=OFF', '-DGGML_CUDA=OFF', '-DGGML_METAL=OFF', '-DLLAMA_OPENSSL=OFF', '-DLLAMA_BUILD_TESTS=OFF', '-DLLAMA_BUILD_EXAMPLES=OFF', '-DLLAMA_BUILD_SERVER=OFF', '-DLLAMA_USE_PREBUILT_UI=OFF', '-DLLAMA_BUILD_UI=OFF'], 'build.log')
run([cmake, '--build', reference_build, '--target', 'llama', '--parallel', '2'], 'build.log')
driver = root / 'reference-driver'
run(['c++', '-std=c++20', '-O3', '-I', reference / 'include', '-I', reference / 'ggml/include', source / 'tests/reference.cpp', '-Wl,--start-group', reference_build / 'src/libllama.a', reference_build / 'ggml/src/libggml.a', reference_build / 'ggml/src/libggml-cpu.a', reference_build / 'ggml/src/libggml-base.a', '-Wl,--end-group', '-pthread', '-ldl', '-o', driver], 'build.log')
revision = subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip()
print('Reference engine built; candidate source commit:', revision)


In [ ]:
models = [
    ('stellan-1-lite', 'https://huggingface.co/OfficialSejeGed/stellan-1-lite-gguf/resolve/94f4d2ffd0990289cda48d9505844f90ac6e7144/stellan-1-lite-q4_k_m.gguf', '84328c452ddef6122b35702f754935ebd7c8bf95a8a9f2b718392cc68968453e'),
    ('stellan-1-max', 'https://huggingface.co/OfficialSejeGed/stellan-1.0-gguf/resolve/fe3b67d162d4b15f0d0b942dd527f7a3382a414d/stellan-1.0-q4_k_m.gguf', '05f00035de8a3893601cec2644f921be240938aab4ff4736ed2f1b6dbcda8dc1')
]
ram = os.sysconf('SC_PHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
try:
    for name, url, digest in models:
        if name == 'stellan-1-max' and ram < 24 * 1024 ** 3:
            print('Skipping Max: use a machine with at least 24 GiB reported RAM for this check')
            continue
        print('Downloading and verifying', name, flush=True)
        weights = root / (name + '.gguf')
        download(url, weights, digest)
        if name == 'stellan-1-max':
            environment['STELLAN_TOKENIZER_GGUF'] = str(weights)
            run([node, '--test', source / 'tests/node-checks.mjs'], 'tests.log')
            environment.pop('STELLAN_TOKENIZER_GGUF', None)
        report_file = results / (name + '.json')
        print('Running five paired CPU probes for', name, flush=True)
        with open(report_file, 'w') as output, open(results / (name + '.log'), 'w') as diagnostic:
            subprocess.run([str(node), str(source / 'compare.mjs'), str(root / 'native-build/stellan-runtime'), str(driver), str(weights), digest, '<s>[INST] Write a Python function add(a, b) that returns their sum. [/INST]', '32', '5', '2', 'no-repack'], check=True, stdout=output, stderr=diagnostic, env=environment)
        report = json.loads(report_file.read_text())
        report['candidateCommit'] = revision
        report['referenceCommit'] = 'cd959bae99ff4bed22bef2604f45241c09bbb8c8'
        report['qualification'] = 'CPU diagnostic only; baseline repacking disabled; not a shipped-app or GPU comparison'
        report_file.write_text(json.dumps(report, indent=2) + '\n')
        print(name, 'completed; matching token streams:', sum(run['identicalGeneratedIds'] for run in report['runs']), '/', len(report['runs']))
        weights.unlink()
finally:
    workspace.cleanup()
print('Finished; download the small JSON reports and logs from stellan-runtime-results')


## What the results mean

Keep the complete reports, including mismatches, CPU time, resident memory, startup and integrity-check time. Different output lengths can make a model look faster, so compare the generated IDs and test the code too. These five paired CPU probes are not enough to remove llama.cpp from the app. They do not exercise Metal/CUDA, activation, encrypted models, desktop packaging or the full agent loop, and they intentionally use the reference's no-repack mode to avoid allocating a second weight copy. Real-device validation with the shipped settings is still required for both models.
